In [1]:
#import json
#process data
import pandas as pd
import numpy as np
# visual
import matplotlib.pyplot as plt
import seaborn as sns
import requests

In [2]:
def get_probe_info(id_probe):
    '''
    Return ('asn_v4' - ASN_code for IPv4, 'country_code', 'is_anchor' - is it anchor ?)
    '''
    req = 'https://atlas.ripe.net/api/v2/probes/' + str(id_probe)
    respond = requests.get(url=req)
    respond = respond.json()

    for i in range(len(respond['tags'])):
        if respond['tags'][i]['name'][:-1]=='system: V':
            eq_type = respond['tags'][i]['name']
            break
        elif respond['tags'][i]['name'][8:]=='Software':
            eq_type = 'Software'
            break
        else:
            eq_type = respond['tags'][1]['name']

    return (respond['asn_v4'], respond['country_code'], respond['is_anchor'], eq_type) #respond['tags'][1]['name'])

In [3]:
# Function to calculate missing values by column
def missing_values_table(df):
        # Total missing values
        mis_val = df.isnull().sum()
        
        # Percentage of missing values
        mis_val_percent = 100 * df.isnull().sum() / len(df)
        
        # Make a table with the results
        mis_val_table = pd.concat([mis_val, mis_val_percent], axis=1)
        
        # Rename the columns
        mis_val_table_ren_columns = mis_val_table.rename(
        columns = {0 : 'Missing Values', 1 : '% of Total Values'})
        
        # Sort the table by percentage of missing descending
        mis_val_table_ren_columns = mis_val_table_ren_columns[
            mis_val_table_ren_columns.iloc[:,1] != 0].sort_values(
        '% of Total Values', ascending=False).round(1)
        
        # Print some summary information
        print ("Your selected dataframe has " + str(df.shape[1]) + " columns.\n"      
            "There are " + str(mis_val_table_ren_columns.shape[0]) +
              " columns that have missing values.")
        
        # Return the dataframe with missing information
        return mis_val_table_ren_columns

In [4]:
def plot_cdf(data, xname):
    plt.rcParams["figure.figsize"] = [7.50, 3.50]
    plt.rcParams["figure.autolayout"] = True

    #data = df_tr[df_tr.hop_1_rtt_mean.notna()].hop_1_rtt_mean.values
    count, bins_count = np.histogram(data, bins=20)
    pdf = count / sum(count)
    cdf = np.cumsum(pdf)
    plt.plot(bins_count[1:], cdf, label="CDF")
    plt.xlabel(xname)
    #plt.legend()
    plt.grid()
    plt.show()

In [5]:
def get_data_by_api(req):
    respond = requests.get(url=req)
    respond = respond.json()
    return pd.DataFrame(respond)

In [6]:
request = 'https://atlas.ripe.net/api/v2/measurements/65489952/results/?format=json'

data = get_data_by_api(request)

In [7]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 27480 entries, 0 to 27479
Data columns (total 26 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   fw                27480 non-null  int64  
 1   mver              26595 non-null  object 
 2   lts               27480 non-null  int64  
 3   dst_name          27480 non-null  object 
 4   af                27480 non-null  int64  
 5   dst_addr          27480 non-null  object 
 6   src_addr          27480 non-null  object 
 7   proto             27480 non-null  object 
 8   ttl               27181 non-null  float64
 9   size              27480 non-null  int64  
 10  result            27480 non-null  object 
 11  dup               27480 non-null  int64  
 12  rcvd              27480 non-null  int64  
 13  sent              27480 non-null  int64  
 14  min               27480 non-null  float64
 15  max               27480 non-null  float64
 16  avg               27480 non-null  float6

In [8]:
data.describe()

,fw,lts,af,ttl,size,dup,rcvd,sent,min,max,avg,msm_id,prb_id,timestamp,group_id,step,stored_timestamp
count,27480.000000,2.748000e+04,27480.0,27181.000000,27480.0,27480.000000,27480.000000,27480.000000,27480.000000,27480.000000,27480.000000,27480.0,2.748000e+04,2.748000e+04,27480.0,27480.0,2.748000e+04
mean,5067.225255,9.859162e+05,4.0,243.369229,32.0,0.000073,3.953130,3.997707,20.407466,23.367079,21.480108,65489952.0,2.957029e+05,1.703710e+09,65489952.0,600.0,1.703711e+09
std,52.126066,8.684116e+06,0.0,2.494279,0.0,0.008531,0.419147,0.049325,18.150977,29.035086,20.801220,0.0,4.292464e+05,5.118159e+04,0.0,0.0,5.092658e+04
min,4790.000000,-1.000000e+00,4.0,233.000000,32.0,0.000000,0.000000,2.000000,-1.000000,-1.000000,-1.000000,65489952.0,1.241000e+03,1.703622e+09,65489952.0,600.0,1.703622e+09
25%,5080.000000,1.800000e+01,4.0,242.000000,32.0,0.000000,4.000000,4.000000,4.143863,5.554709,4.546471,65489952.0,2.649200e+04,1.703666e+09,65489952.0,600.0,1.703666e+09
50%,5080.000000,2.700000e+01,4.0,244.000000,32.0,0.000000,4.000000,4.000000,16.128305,17.570439,16.755904,65489952.0,5.344400e+04,1.703711e+09,65489952.0,600.0,1.703710e+09
75%,5080.000000,4.400000e+01,4.0,245.000000,32.0,0.000000,4.000000,4.000000,30.741844,35.653585,32.415197,65489952.0,1.000807e+06,1.703754e+09,65489952.0,600.0,1.703755e+09
max,5080.000000,8.407432e+07,4.0,247.000000,32.0,1.000000,4.000000,4.000000,418.475788,1673.842511,571.472724,65489952.0,1.006726e+06,1.703799e+09,65489952.0,600.0,1.704225e+09


In [9]:
len(data)

27480

In [10]:
missing_values_table(data)

Your selected dataframe has 26 columns.
There are 2 columns that have missing values.


,Missing Values,% of Total Values
mver,885,3.2
ttl,299,1.1


In [11]:
data.shape

(27480, 26)

In [12]:
data.loc[0].result

[{'rtt': 44.231098}, {'rtt': 44.258342}, {'rtt': 43.798279}, {'rtt': 43.77892}]

In [13]:
data.loc[0].avg

44.01665975

In [14]:
data.columns

Index(['fw', 'mver', 'lts', 'dst_name', 'af', 'dst_addr', 'src_addr', 'proto',
       'ttl', 'size', 'result', 'dup', 'rcvd', 'sent', 'min', 'max', 'avg',
       'msm_id', 'prb_id', 'timestamp', 'msm_name', 'from', 'type', 'group_id',
       'step', 'stored_timestamp'],
      dtype='object')

In [15]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 27480 entries, 0 to 27479
Data columns (total 26 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   fw                27480 non-null  int64  
 1   mver              26595 non-null  object 
 2   lts               27480 non-null  int64  
 3   dst_name          27480 non-null  object 
 4   af                27480 non-null  int64  
 5   dst_addr          27480 non-null  object 
 6   src_addr          27480 non-null  object 
 7   proto             27480 non-null  object 
 8   ttl               27181 non-null  float64
 9   size              27480 non-null  int64  
 10  result            27480 non-null  object 
 11  dup               27480 non-null  int64  
 12  rcvd              27480 non-null  int64  
 13  sent              27480 non-null  int64  
 14  min               27480 non-null  float64
 15  max               27480 non-null  float64
 16  avg               27480 non-null  float6

In [16]:
data.groupby('prb_id').avg.count()

prb_id
1241       295
3891       295
4261       295
10734      295
10869      295
          ... 
1006284    295
1006351    295
1006531    295
1006711    295
1006726    295
Name: avg, Length: 95, dtype: int64

In [18]:
data.prb_id.nunique()

95